##### 简单注意力机制

In [1]:
import torch
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your
        [0.55, 0.87, 0.66], # journey
        [0.57, 0.85, 0.64], # starts
        [0.22, 0.58, 0.33], # with
        [0.77, 0.25, 0.10], # one
        [0.05, 0.80, 0.55] # step
    ]
)

In [2]:
# 第二个输入词元作为查询向量
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


##### 进行归一化处理

In [3]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


##### 简单的 softmax 实现，可能会遇到数值稳定性问题，比如溢出和下溢。实践中建议使用 softmax 的 pytorch 实现

In [4]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [5]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


##### 计算上下文向量

In [6]:
# 第二个输入词元作为查询向量
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print("Context vector:", context_vec_2)

Context vector: tensor([0.4419, 0.6515, 0.5683])


##### 计算所有输入词元的注意力权重

In [7]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [8]:
# 使用矩阵乘法比 for 循环更快
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [9]:
# 对每一行进行归一化
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [10]:
# 验证总和为 1
row_2_sum = sum([0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [11]:
# 用注意力权重通过矩阵乘法计算出所有上下文向量
all_context_vecs = attn_weights @ inputs
print("All context vectors:", all_context_vecs)

All context vectors: tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [12]:
# 第二行完全匹配
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


##### 逐步计算注意力权重

In [13]:
import torch

# 创建输入张量，表示一个包含6个词的序列，每个词由3维向量表示
# 这是一个6x3的矩阵，每一行代表一个词的嵌入向量(embedding vector)
# 例如：第一行[0.43, 0.15, 0.89]是单词"Your"的嵌入向量表示
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your - 第1个词的3维嵌入向量
        [0.55, 0.87, 0.66], # journey - 第2个词的3维嵌入向量
        [0.57, 0.85, 0.64], # starts - 第3个词的3维嵌入向量
        [0.22, 0.58, 0.33], # with - 第4个词的3维嵌入向量
        [0.77, 0.25, 0.10], # one - 第5个词的3维嵌入向量
        [0.05, 0.80, 0.55]  # step - 第6个词的3维嵌入向量
    ]
)

In [14]:
x_2 = inputs[1]  # 获取第二个输入元素(索引为1)，即"journey"对应的向量 [0.55, 0.87, 0.66]
d_in = inputs.shape[1]  # 输入嵌入维度 d_in=3，表示每个词由3维向量表示
d_out = 2  # 输出维度 d_out=2，经过线性变换后的向量维度，通常是为了降维或提取更有意义的特征

In [15]:
# 初始化权重矩阵，设置随机种子确保结果可重现
# 在实际应用中，这些权重会在训练过程中通过反向传播进行学习和更新
# 使用 torch.manual_seed(123) 确保每次运行代码时生成的随机数相同，便于调试和复现结果
# 使用 torch.nn.Parameter 将张量标记为模型参数，使其可以被优化器更新
# requires_grad=False 表示在当前示例中不进行梯度计算，实际训练时应设为 True
torch.manual_seed(123)
# 查询权重矩阵(W_query): 用于计算查询向量(Q)，形状为(d_in, d_out)即(3, 2)
# 查询向量用于衡量当前词对序列中其他词的关注程度
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# 键权重矩阵(W_key): 用于计算键向量(K)，形状为(d_in, d_out)即(3, 2)
# 键向量用于表示词的特征，供其他词进行匹配和比较
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# 值权重矩阵(W_value): 用于计算值向量(V)，形状为(d_in, d_out)即(3, 2)
# 值向量包含实际的信息内容，在注意力机制中会被加权求和
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [16]:
# 对单个输入向量 x_2 进行线性变换，得到对应的查询、键、值向量
# 使用矩阵乘法(@)将输入向量与相应的权重矩阵相乘
query_2 = x_2 @ W_query  # 查询向量，形状为(2,)，用于衡量当前词对其他词的关注度
keys_2 = x_2 @ W_key    # 键向量，形状为(2,)，用于表示词的特征供其他词匹配
values_2 = x_2 @ W_value # 值向量，形状为(2,)，用于加权求和得到最终的上下文表示

In [17]:
print("Query vector:", query_2)  # 输出查询向量

Query vector: tensor([0.4306, 1.4551])


In [18]:
keys = inputs @ W_key    # 所有输入词的键向量，形状为(6, 2)
values = inputs @ W_value # 所有输入词的值向量，形状为(6, 2)
print("key.shape:", keys.shape)
print("value.shape:", values.shape)

key.shape: torch.Size([6, 2])
value.shape: torch.Size([6, 2])


In [19]:
# 计算注意力分数(attention scores)矩阵
keys_2 = keys[1]  # 注意：Python从0开始进行检索
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [20]:
attn_scores_2 = query_2 @ keys.T  # 给定query的全部注意力分数
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [21]:
# 计算注意力权重(attention weights)
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [22]:
# 计算上下文向量(context vector)
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


#### 代码清单 3-1 一个简化的自注意力Python类

In [23]:
import torch.nn as nn
class SelfAttention_V1(nn.Module):
    # 构造函数，接收输入维度d_in和输出维度d_out作为参数
    def __init__(self, d_in, d_out):
        # 调用父类nn.Module的构造函数
        super().__init__()
        # 使用Parameter定义查询、键、值的权重矩阵，这些参数会自动注册到模型中
        # rand函数生成[0,1)区间内的均匀分布随机数，初始化权重矩阵
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    # 前向传播函数，接收输入张量x
    def forward(self, x):
        # 计算键向量：输入x与键权重矩阵相乘
        keys = x @ self.W_key
        # 计算查询向量：输入x与查询权重矩阵相乘
        queries = x @ self.W_query
        # 计算值向量：输入x与值权重矩阵相乘
        values = x @ self.W_value
        # 计算注意力分数：查询向量与键向量的转置相乘
        attn_scores = queries @ keys.T
        # 对注意力分数进行缩放并应用softmax函数得到注意力权重
        # keys.shape[-1]获取键向量的最后一维大小，即d_out
        # 除以sqrt(d_k)进行缩放，dim=-1表示在最后一个维度上进行softmax
        attn_weight = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5), dim=-1
        )
        # 使用注意力权重对值向量进行加权求和得到上下文向量
        context_vec = attn_weight @ values
        # 返回计算得到的上下文向量
        return context_vec

In [24]:
# 使用SelfAttention_V1类进行自注意力计算
torch.manual_seed(123)
sa_v1 = SelfAttention_V1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


#### 代码清单 3-2 一个使用PyTorch线性层的自注意力类

In [25]:
# 定义另一个自注意力机制版本，使用PyTorch内置的Linear层
class SelfAttention_V2(nn.Module):
    # 构造函数，接收输入维度d_in、输出维度d_out和是否使用偏置项qkv_bias作为参数
    def __init__(self, d_in, d_out, qkv_bias=False):
        # 调用父类nn.Module的构造函数
        super().__init__()
        # 使用PyTorch内置的Linear层定义查询、键、值的线性变换
        # Linear层会自动创建可学习的权重和偏置参数
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # 前向传播函数，接收输入张量x
    def forward(self, x):
        # 通过Linear层计算键向量
        keys = self.W_key(x)
        # 通过Linear层计算查询向量
        queries = self.W_query(x)
        # 通过Linear层计算值向量
        values = self.W_value(x)
        # 计算注意力分数：查询向量与键向量的转置相乘
        attn_scores = queries @ keys.T
        # 对注意力分数进行缩放并应用softmax函数得到注意力权重
        attn_weight = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5), dim=-1
        )
        # 使用注意力权重对值向量进行加权求和得到上下文向量
        context_vec = attn_weight @ values
        # 返回计算得到的上下文向量
        return context_vec

In [26]:
# 使用SelfAttention_V2类进行自注意力计算
torch.manual_seed(789)
sa_v2 = SelfAttention_V2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


##### 因果注意力的掩码实现

In [27]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(
    attn_scores / (keys.shape[-1] ** 0.5), dim=-1
)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [28]:
context_length = attn_scores.shape[0]

# 创建下三角矩阵作为简单掩码（方法1）
# torch.tril会保留矩阵的下三角部分（包括对角线），其余位置置0
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [29]:
# 将注意力权重与简单掩码相乘，屏蔽未来位置的信息
mask_simple *= attn_weights
print(mask_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [30]:
# 对每行进行归一化，使每行的权重和为1
row_sums = mask_simple.sum(dim=-1, keepdim=True)
mask_simple_norm = mask_simple / row_sums
print(mask_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [31]:
# 创建上三角矩阵作为掩码（方法2，推荐方法）
# torch.triu会保留矩阵的上三角部分（diagonal=1表示从主对角线上方开始）
# 这种方式更适合用于因果注意力，因为可以将未来位置设置为负无穷
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

# 使用掩码填充注意力分数，将未来位置的分数设为负无穷
# 这样在经过softmax后，这些位置的权重就会接近0
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [32]:
# 对掩码后的注意力分数应用softmax，得到最终的因果注意力权重
# 现在每个位置只能关注到它之前和当前位置的信息
attn_weights = torch.softmax(
    masked / (keys.shape[-1] ** 0.5), dim=-1
)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


##### 利用 dropout 掩码额外的注意力权重

In [33]:
torch.manual_seed(123)
dropout = nn.Dropout(p=0.5) # 选择使用50%的dropout率
example = torch.ones(6, 6) # 在这里创建一个全1矩阵
print(dropout(example))

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [34]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.8966, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4921, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4350, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


##### 实现一个简化的因果注意力类

In [35]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 两个输入，每个输入有6个词元，每个词元的嵌入维度为3

torch.Size([2, 6, 3])


#### 代码清单 3-3 一个简化的因果注意力类

In [36]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        # 定义查询、键、值的线性变换层
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # 定义dropout层
        self.dropout = nn.Dropout(dropout)

        # 创建并注册上三角掩码缓冲区，防止未来信息泄露
        # torch.triu创建上三角矩阵，diagonal=1表示从主对角线上方开始
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        # 获取输入张量的形状信息
        b, num_tokens, d_in = x.shape

        # 计算键、查询、值向量
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 计算注意力分数: 查询向量与键向量的转置相乘
        # 使用transpose(1, 2)对键向量进行转置操作
        attn_scores = queries @ keys.transpose(1, 2)

        # 应用因果掩码，将未来位置的注意力分数设为负无穷
        # 这样在softmax后这些位置的权重会接近0
        attn_scores.masked_fill(
            self.mask.bool()[:num_tokens, :num_tokens],  # 只使用前num_tokens行和列的掩码
            -torch.inf
        )

        # 对注意力分数进行缩放并应用softmax和dropout得到注意力权重
        attn_weights = self.dropout(torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5), dim=-1  # 缩放因子为维度的平方根
        ))

        # 使用注意力权重对值向量进行加权求和得到上下文向量
        context_vec = attn_weights @ values
        return context_vec

In [37]:
torch.manual_seed(123)
context_length = batch.shape[1]
# 创建因果自注意力实例
ca = CausalSelfAttention(d_in, d_out, context_length, dropout=0.0)
# 对批处理数据应用因果注意力
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])
